## **Assignment 02 on Training CNNs**

### **Foundational Setup** 
This code handles fixing the random seeds for reproducibility, calculating the exact channel means and standard deviations from the raw CIFAR-10 training data, applying those statistics to normalize the dataset, and splitting it into the required 45k/5k subsets.

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader
import numpy as np
import random

# 1. Fix random seeds for complete reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
root_dir = './data'

# 2. Load raw CIFAR-10 strictly to get the indices for the split
raw_train_set = torchvision.datasets.CIFAR10(root=root_dir, train=True, download=False)

# Split into 45k and 5k to get the exact indices for the training subset
train_size = 45000
val_size = 5000
raw_train_subset, raw_val_subset = random_split(
    raw_train_set, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

# 3. Extract ONLY the 45k training images to calculate statistics
# raw_train_subset.indices safely extracts just the 45,000 training images
train_45k_data = raw_train_set.data[raw_train_subset.indices]

cifar_mean = train_45k_data.mean(axis=(0, 1, 2)) / 255.0
cifar_std = train_45k_data.std(axis=(0, 1, 2)) / 255.0

print(f"Calculated 45k Training Mean (RGB): {cifar_mean}")
print(f"Calculated 45k Training Std (RGB): {cifar_std}")

# 4. Define the transform using ONLY the 45k computed statistics
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean.tolist(), std=cifar_std.tolist())
])

# 5. Load the datasets again, this time with the normalization transform applied
full_train_dataset_transformed = torchvision.datasets.CIFAR10(root=root_dir, train=True, download=False, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root=root_dir, train=False, download=False, transform=transform)

# 6. Re-apply the EXACT SAME split (using the same seed) so our transformed datasets match the indices
train_dataset, val_dataset = random_split(
    full_train_dataset_transformed, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42) 
)

# 7. Initialize DataLoaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train set size: {len(train_dataset)} | Val set size: {len(val_dataset)} | Test set size: {len(test_dataset)}")

Calculated 45k Training Mean (RGB): [0.49165571 0.48233405 0.44667923]
Calculated 45k Training Std (RGB): [0.24708116 0.24350869 0.26158066]
Train set size: 45000 | Val set size: 5000 | Test set size: 10000
